Registry **design decisions #1–#7** are **resolved** in **`pipeline_spec.md`**. This notebook is a **short status table**; **contradiction checking** remains a **deferred** pipeline component (spec’d to use the graph shape below).

Documentation updates (done)
asp/wfm/Agent_WFM.md

Patch merge: Clarifies that the authoritative list is the text as shown in the confirmation package (PASS / REWRITE as displayed, then patch + omits), not “Agent 2 raw only.”
Open issues: Notes Agent 3’s machine-pinned line format and adds a WFM test harness row (Gemini runners, Agent 4 + wfm_agent4_common, interactive, loopback script + test_sets/README.md).
pipeline_spec.md

WFM step: Footnote pointing at test_sets/scripts/ and test_sets/README.md for the harness (not the product orchestrator).
Registry: **Graph-friendly shape** (A3/B3/C1/D3); **Rules store ↔ registry alignment**; **Failure / commit contract** (`bundles/*.json`, user partial commit, late durable registry). **Registry Design** intro lists persistence files; **Consistency checking** still deferred.
asp/wfm/prompts/README.md

Agent 3 maintenance note: outputs are machine-pinned for merge/tooling.
After Agent 4 when loop-back is not entered (the “yes” path)
Aligned across Agent_WFM.md and pipeline_spec.md:

User accepts the confirmation package (no Agent 4 LLM, or Agent 4 + merge already done and accepted).
Orchestration hands off to step 3: Registry agent receives the **structured** WFM→registry payload (per-line **`statement_nl`**, verdicts, bundle id, timestamps—see **`pipeline_spec.md`** *WFM → registry handoff*), not prose alone.
Registry agent: search (embeddings) → extract gaps → resolve (mostly auto) → populate **per line**; then formalizer / Z3 / critic as later steps.
So in your terminology, “the registry” is the next stage after a successful WFM confirmation without (or after finishing) loop-back. The loop-back harness you ran exercises merge → A1–3 only; it does not implement registry — that is still specified, not built.

## What the plans already say (structure & workflow)

Workflow (pipeline_spec.md step 3): semantic search first, extraction only for gaps, search results authoritative, resolve with side-by-side NL and ID substitution, populate and pass registry context to the formalizer.

Per-entry shape: ID, kind (sort | constant | function), name, signature / parent sort / members as applicable, source rule, NL description, embedding of description + name.

Infrastructure (planned): BAAI/bge-base-en-v1.5, FAISS, registry.json + rules.json.

Deferred: explicit consistency / contradiction component, now explicitly linked to graph traversal in the spec.













## Design decisions — status (canonical detail: **`pipeline_spec.md`**)

| # | Topic | Status | Resolution |
|---|--------|--------|------------|
| 1 | **ID policy** | **Resolved** | Immutable registry + rule IDs; no silent reuse (tombstone or equivalent). See **Graph-friendly shape** in `pipeline_spec.md`. |
| 2 | **Rule granularity** | **Resolved** | One **`rule_id`** per accepted **confirmation line**; shared **`bundle_id`** / **`wfm_acceptance_id`** per user acceptance; **`line_index`** for order. |
| 3 | **WFM → registry handoff** | **Resolved** | Structured payload: bundle fields (**`bundle_id`**, **`user_original_input`**, **`confirmation_package_style_a`**, **`orchestration_run_id`**, **`wfm_pipeline_timestamps`**, optional **`provider_model`** + **`wfm_compound_operator_limit`**) and per-line fields (**`statement_nl`**, **`agent3_verdict`**, **`scope_report`/`diff_report`**, **`agent2_line_text`**). **OUT_OF_SCOPE** lines stay in payload; skip formalization. Full tables in **`pipeline_spec.md`** — *WFM → registry handoff*. |
| 4 | **Explicit graph edges** | **Resolved** | **A3** draft + committed rule→entry links; **B3** labeled edges (`rule_id`, `entry_id`, closed `relationship` enum from pipeline artifacts); **C1** authoritative **`source_rule`** on entries + rule-side mirror updated **same commit**; **D3** hybrid stored vs computed **entry ↔ entry**. Canonical: **`pipeline_spec.md`** *Graph-friendly shape*; concrete behavior: **`registry_graph_edges_when_built_concrete.md`**; options narrative: **`registry_graph_edges_design_choices.md`**. |
| 5 | **Dual retrieval** | **Resolved by spec** | Structural fields queryable; FAISS for embedding—already `pipeline_spec.md` **Registry Design**. |
| 6 | **Rules store vs registry** | **Resolved** | Canonical **`committed_edges`** on each rule row; **`wfm_pipeline_timestamps`** + **`orchestration_run_id`** from handoff; **full handoff** at **`bundles/{bundle_id}.json`** next to `registry.json` / `rules.json` (**no per-rule `handoff_ref`**; override reserved for future migration only); lightweight **bundle** record; **rule** semantic embeddings **deferred until contradiction / candidate-search**; **`validate_alignment`** + ops backup for drift. **`pipeline_spec.md`** — *Rules store ↔ registry alignment (resolved)*; plain language: **`rules_store_registry_alignment_explained.md`**. |
| 7 | **Failure / commit contract** | **Resolved** | **Late** production registry; **in-memory** draft v1 (no `runs/` scratch unless added later); **`bundles/{bundle_id}.json`** on accept + **`bundles/{bundle_id}.pipeline.json`** for status; **user-directed** full vs **partial** commit; **new `bundle_id`** for retry of uncommitted lines; **single writer**; **idempotent** commit; recovery via **backup** + **forward** supersede. **`pipeline_spec.md`** — *Failure / commit contract (resolved)*. |

## What still needs discussion (remaining)

*(None — design table above; implementation follows `pipeline_spec.md`.)*

**Bottom line:** Design decisions **#1–#7** — **`pipeline_spec.md`**. **Persistence / Phase 1 / gaps:** **`registry_persistence_v1.md`**. **Dev milestones (M0–M6):** **`development_plan_registry_stage_v1.md`**.